# Ukrainian Linguistic Decolonization & Reasoning (ULDR)
## Phase 3.6: Pilot Canary Fine-Tune on Google Gemma 3 4B-it

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/learn-ukrainian/learn-ukrainian.github.io/blob/main/scripts/projects/open_model_data/pilot_canary_gemma3_4b_colab.ipynb)

This notebook fine-tunes **Google Gemma 3 4B-it** on the 200-item Phase 3.6 pilot canary dataset with a 15% general Ukrainian replay buffer, evaluates the directional safety gates (Calque Elimination $\ge 90\%$, Harmful-Edit Rate $\le 1.0\%$, NLP margin $\le 1.5\%$) on the 900-case evaluation suite, and saves the resulting adapter and evaluation outputs.

### Pipeline & Environment Architecture
- **Repository & CI**: Runs deterministic offline contract & schema verification (`v4_pilot_canary_evaluation.py`) without requiring GPU or PyTorch in CI.
- **Downstream GPU Execution**: This notebook runs on external GPU hardware (Free Google Colab T4 or A100) to execute authentic QLoRA fine-tuning and inference.
- **Public Dataset Hub**: Datasets and evaluation suites are publicly hosted at [`https://huggingface.co/krisztiankoos/uldr-canary-artifacts`](https://huggingface.co/krisztiankoos/uldr-canary-artifacts).

### Hardware Requirement
- Free Google Colab T4 GPU (or A100/V100 on Colab Pro).
- Estimated run time: ~5–7 minutes.

In [ ]:
# Step 1: Install fine-tuning and evaluation dependencies
!pip install -q --upgrade transformers peft accelerate bitsandbytes huggingface_hub scipy jsonschema safetensors

In [ ]:
# Step 2: Hugging Face Authentication
# Note: Google Gemma 3 is a gated model. Accepting terms on huggingface.co/google/gemma-3-4b-it is required.
import os

from huggingface_hub import HfApi, hf_hub_download, login

try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    hf_token = input("Enter Hugging Face Token: ").strip()

login(token=hf_token)
api = HfApi(token=hf_token)
print("Authenticated as:", api.whoami()["name"])

In [ ]:
# Step 3: Download Canary Dataset, Replay Buffer, and 900-Case Evaluation Suite
# Note: krisztiankoos/uldr-canary-artifacts is a public Hugging Face repository.
repo_id = "krisztiankoos/uldr-canary-artifacts"

train_file = hf_hub_download(repo_id=repo_id, filename="pilot_canary_train_200.jsonl")
replay_file = hf_hub_download(repo_id=repo_id, filename="pilot_canary_replay_buffer_30.jsonl")
eval_cases_file = hf_hub_download(repo_id=repo_id, filename="pilot_canary_eval_cases.jsonl")

print(f"Successfully downloaded datasets from public hub: {repo_id}")

In [ ]:
# Step 4: Load Gemma 3 4B-it Model with 4-bit Quantization
import torch
from peft import LoraConfig, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_id = "google/gemma-3-4b-it"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(model_id, token=hf_token)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    token=hf_token,
)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# Step 5: Prepare Tokenized Training Dataset
import json

from torch.utils.data import DataLoader, Dataset


class CanaryDataset(Dataset):
    def __init__(self, train_path, replay_path, tokenizer, max_length=1024):
        self.items = []
        with open(train_path, encoding="utf-8") as f:
            for line in f:
                if line.strip():
                    self.items.append(json.loads(line))
        with open(replay_path, encoding="utf-8") as f:
            for line in f:
                if line.strip():
                    self.items.append(json.loads(line))
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        item = self.items[idx]
        p = item.get("query") or item.get("instruction") or ""
        r = item.get("final_response") or item.get("response") or ""
        text = f"<start_of_turn>user\n{p}<end_of_turn>\n<start_of_turn>model\n{r}<end_of_turn>"
        enc = self.tokenizer(text, truncation=True, max_length=self.max_length, padding="max_length", return_tensors="pt")
        input_ids = enc["input_ids"][0]
        attention_mask = enc["attention_mask"][0]
        labels = input_ids.clone()
        labels[labels == self.tokenizer.pad_token_id] = -100
        return {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels}

dataset = CanaryDataset(train_file, replay_file, tokenizer)
loader = DataLoader(dataset, batch_size=2, shuffle=True)
print(f"Total training records: {len(dataset)} across {len(loader)} steps per epoch")

In [ ]:
# Step 6: Fine-Tune with AdamW and Cosine Learning Rate Decay
from torch.optim import AdamW
from transformers import get_cosine_schedule_with_warmup

epochs = 3
total_steps = len(loader) * epochs
optimizer = AdamW(model.parameters(), lr=2e-4, weight_decay=0.01)
scheduler = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps=10, num_training_steps=total_steps)

model.train()
step_losses = []
for _epoch in range(epochs):
    for _step, batch in enumerate(loader):
        input_ids = batch["input_ids"].cuda()
        attention_mask = batch["attention_mask"].cuda()
        labels = batch["labels"].cuda()

        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()

        step_losses.append(loss.item())
        if len(step_losses) % 10 == 0 or len(step_losses) == 1:
            print(f"Step {len(step_losses)}/{total_steps} - Loss: {loss.item():.4f}")

initial_loss = step_losses[0]
converged_loss = sum(step_losses[-10:]) / 10
reduction_pct = ((initial_loss - converged_loss) / initial_loss) * 100
print(f"Initial loss: {initial_loss:.4f} -> Converged loss: {converged_loss:.4f} ({reduction_pct:.2f}% reduction)")

In [ ]:
# Step 7: Save LoRA Adapter Weights and Training Log
output_dir = "canary_artifacts"
os.makedirs(output_dir, exist_ok=True)

model.save_pretrained(output_dir)
print(f"Adapter saved to {output_dir}")

training_log_path = f"{output_dir}/pilot_canary_training_log.jsonl"
with open(training_log_path, "w", encoding="utf-8") as f:
    for idx, loss_val in enumerate(step_losses, 1):
        rec = {
            "step": idx,
            "loss": round(loss_val, 4),
            "learning_rate": round(scheduler.get_last_lr()[0], 8),
        }
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

In [ ]:
# Step 8: Directional Safety Gates Evaluation on 900 Cases
from scipy.stats import beta

print("Evaluating directional safety gates on fine-tuned Gemma 3 4B-it...")
model.eval()

eval_cases = []
with open(eval_cases_file, encoding="utf-8") as f:
    for line in f:
        if line.strip():
            eval_cases.append(json.loads(line))

calque_cases = [c for c in eval_cases if c.get("suite") == "calque_elimination"]
safety_cases = [c for c in eval_cases if c.get("suite") == "clean_control_safety"]
nlp_cases = [c for c in eval_cases if c.get("suite") == "general_nlp_benchmark"]

print(f"Evaluation suites: {len(calque_cases)} Calque | {len(safety_cases)} Clean Controls | {len(nlp_cases)} General NLP")

def generate_prediction(prompt_text, max_new_tokens=128):
    prompt_formatted = f"<start_of_turn>user\n{prompt_text}<end_of_turn>\n<start_of_turn>model\n"
    inputs = tokenizer(prompt_formatted, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    gen_text = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return gen_text.strip()

# 1. Calque Elimination Gate (target >= 90.0%)
calque_eliminated = 0
for c in calque_cases:
    pred = generate_prediction(c["input_prompt"], max_new_tokens=64)
    targ = c["target_term"].lower()
    repl = c["expected_replacement"].lower()
    if targ not in pred.lower() and repl in pred.lower():
        calque_eliminated += 1

calque_elim_rate = (calque_eliminated / len(calque_cases)) * 100

# 2. Harmful Edit Rate on Clean Controls (target <= 1.0%)
harmful_edits = 0
for c in safety_cases:
    pred = generate_prediction(c["input_prompt"], max_new_tokens=64)
    term = c["target_term"].lower()
    if term not in pred.lower():
        harmful_edits += 1

harmful_rate = (harmful_edits / len(safety_cases)) * 100
k = harmful_edits
n = len(safety_cases)
cp_upper_bound = 100.0 if k == n else float(beta.ppf(0.95, k + 1, n - k)) * 100

# 3. General NLP Non-Inferiority Margin (target <= 1.5%)
nlp_correct = 0
for c in nlp_cases:
    pred = generate_prediction(c["input_prompt"], max_new_tokens=32)
    expected = c.get("expected_answer", "").strip().lower()
    if expected and expected in pred.lower():
        nlp_correct += 1

nlp_acc = (nlp_correct / len(nlp_cases)) * 100
baseline_nlp_acc = 82.00  # Baseline Gemma 3 4B-it accuracy on Eval-UA-tion 1.0 subset
nlp_margin = max(0.0, baseline_nlp_acc - nlp_acc)

calque_passed = calque_elim_rate >= 90.0
safety_passed = cp_upper_bound <= 1.0
nlp_passed = nlp_margin <= 1.5
all_passed = calque_passed and safety_passed and nlp_passed

print("\n=== DIRECTIONAL SAFETY GATES REPORT ===")
print(f"1. Calque Elimination Rate: {calque_elim_rate:.2f}% (Threshold >= 90.0%) -> {'PASS' if calque_passed else 'FAIL'}")
print(f"2. Harmful Edit Rate:       {harmful_rate:.2f}% (95% UB: {cp_upper_bound:.2f}%, Threshold <= 1.0%) -> {'PASS' if safety_passed else 'FAIL'}")
print(f"3. General NLP Margin:      {nlp_margin:.2f}% (Threshold <= 1.5%) -> {'PASS' if nlp_passed else 'FAIL'}")
print(f"Overall Safety Gate Verdict: {'CANARY_PILOT_PASSED' if all_passed else 'CANARY_PILOT_FAILED'}")

eval_results = {
    "base_model": model_id,
    "evaluation_suite": "pilot_canary_eval_cases_900",
    "calque_elimination_rate_pct": round(calque_elim_rate, 2),
    "harmful_edit_rate_pct": round(harmful_rate, 2),
    "clopper_pearson_95_upper_bound_pct": round(cp_upper_bound, 2),
    "general_nlp_margin_pct": round(nlp_margin, 2),
    "verdict": "CANARY_PILOT_PASSED" if all_passed else "CANARY_PILOT_FAILED",
}

eval_results_file = f"{output_dir}/evaluation_results.json"
with open(eval_results_file, "w", encoding="utf-8") as f:
    json.dump(eval_results, f, indent=2, ensure_ascii=False)
print(f"Saved evaluation results to {eval_results_file}")

In [ ]:
# Step 9: Upload Trained Adapter, Training Log, and Evaluation Results to Hugging Face
api.upload_folder(
    folder_path=output_dir,
    repo_id=repo_id,
    repo_type="model",
    commit_message="Upload trained Gemma 3 4B LoRA adapter and evaluation results",
)
print(f"[SUCCESS] All canary training & evaluation artifacts pushed to https://huggingface.co/{repo_id}")